In [1]:
# 1. Imports and setup
from romansh_lemmatizer import Lemmatizer, Lemma
from collections import Counter
import pandas as pd
from pathlib import Path
from datasets import load_dataset
import re
from collections import Counter

mediomatix = load_dataset("ZurichNLP/mediomatix", split="train[:5000]")
idioms = ["rm-sursilv", "rm-sutsilv", "rm-surmiran", "rm-puter", "rm-vallader"]


c:\Users\Dominic-Asus\Rumantsch_Projekt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Inspection
"""
inspection_data = mediomatix[0]
for k, v in inspection_data.items():
    print(k, v)
print()
"""

'\ninspection_data = mediomatix[0]\nfor k, v in inspection_data.items():\n    print(k, v)\nprint()\n'

In [3]:

# 3. Build corpus
corpus = {k:[] for k in idioms}
for row in mediomatix:
    for idiom in idioms:
        if row.get(idiom):
            sent = row[idiom]
            sent = re.sub(r"</?strong>", " ", sent) # Remove <strong> tags
            sent = re.sub(r'["“”«»‚‘’„][^"“”«»‚‘’„]+["“”«»‚‘’„]', " ", sent) # Remove quoted substrings (straight, angled, curly quotes)
            corpus[idiom].append(sent)

for k, v in corpus.items():
    print(k, len(v))

rm-sursilv 4864
rm-sutsilv 4562
rm-surmiran 1036
rm-puter 4846
rm-vallader 4816


In [4]:
# 4. Initialize lemmatizers (specific idioms)
lemmatizers = {idiom: Lemmatizer(idiom=idiom) for idiom in idioms}

In [5]:

# 5. Build per-idiom lemma + translation counters
corpus_lemma_counter = {k: Counter() for k in idioms}

for idiom, sentences in corpus.items():
    for sent in sentences:
        doc = lemmatizers[idiom](sent)
        for t in doc.tokens:
            lemma_objs = list(t.lemmas.keys())
            if lemma_objs:
                lemma_obj = lemma_objs[0]
                lemma_text = lemma_obj.text
                lemma_trans = lemma_obj.translation_de
            else:
                lemma_text = t.text
                lemma_trans = None

            corpus_lemma_counter[idiom][(lemma_text, lemma_trans)] += 1

In [20]:
# 6. Define filtering func
def filter_lemmas(counter):
    """Filter out short or malformed lemmas."""
    return {
        (lemma, trans): freq
        for (lemma, trans), freq in counter.items()
        if lemma and len(lemma) > 1 and "//" not in lemma
    }

In [21]:
# 7. Pretty print results
for idiom, counter in corpus_lemma_counter.items():
    filtered = filter_lemmas(counter)
    if not filtered:
        continue

    # Create DataFrame
    df = pd.DataFrame(
        [(lemma, trans or "", freq) for (lemma, trans), freq in Counter(filtered).most_common(20)],
        columns=["Lemma", "German", "Freq"]
    )

    # Sort by frequency descending (optional)
    df = df.sort_values("Freq", ascending=False).reset_index(drop=True)

    print(f"\n===== {idiom} =====")
    display(df)


===== rm-sursilv =====


,Lemma,German,Freq
0,il,der,998
1,esser,sein,970
2,da,aus,958
3,in,eins,845
4,la,die,776
5,ils,cf. il,660
6,haver,haben,615
7,ina,eine,515
8,discuorer,sprechen,514
9,che,null,491



===== rm-sutsilv =====


,Lemma,German,Freq
0,igl,es,1502
1,la,null,1253
2,bu(g)agear,cf. uagear,1120
3,(da)schnu(v)ar,abknöpfen,1083
4,egn,Eins,725
5,easser,sich aufhalten,714
6,ver,besitzen,707
7,igls,null,630
8,catar,Katarrh,567
9,las,null,545



===== rm-surmiran =====


,Lemma,German,Freq
0,igl,den,287
1,la,die,253
2,da,als,216
3,esser,liegen,170
4,igls,die,152
5,veir,haben,149
6,tgi,als,149
7,en,Inn,148
8,cun,anhand,146
9,las,sie,142



===== rm-puter =====


,Lemma,German,Freq
0,ün,Eins,1374
1,da,mit,1198
2,il,das (vor Konsonant),1081
3,la,das (vor Konsonant),979
4,avair,Lust haben,761
5,in,auf,628
6,ils,die,609
7,esser,im Wald sein,606
8,las,die,553
9,cun,mit,462



===== rm-vallader =====


,Lemma,German,Freq
0,da,über,1194
1,il,das (vor Konsonanten),1053
2,la,La,948
3,vulair,mögen (wollen),822
4,ün,ein,813
5,in,an,609
6,ils,die,577
7,esser,an etw gewöhnt sein,551
8,üna,eine,525
9,las,die,518
